# Testing & Training — Pipeline Huấn Luyện YOLO11n + CBAM (Dataset V2)

> **Lưu ý trước khi chạy:**
> 1. **Runtime GPU:** Chọn **T4 GPU** (`Runtime` -> `Change runtime type` -> `T4 GPU`).
> 2. **Lối tắt Drive:** Hãy tạo lối tắt (Shortcut) thư mục chung `KLTN-2026-testingNtraining` (hoặc `KLTN_38`) vào `My Drive` của bạn trước khi chạy.
> 3. **Cấu trúc lưu kết quả:** Toàn bộ weights, logs và biểu đồ so sánh sẽ được lưu trực tiếp vào thư mục mô hình tương ứng trên Google Drive (`runs_exp2_{RATIO}/{variant}`) mà không bị lồng thêm cấp con `train/`.

## Bước 1️⃣: Kết Nối Google Drive & Setup Môi Trường

In [ ]:
import os
import sys
from pathlib import Path
from google.colab import drive

# 1. Kết nối Google Drive
drive.mount('/content/drive')

# 2. Nhận diện thư mục làm việc chung trên Google Drive
CANDIDATES = [
    "/content/drive/MyDrive/KLTN-2026-testingNtraining",
    "/content/drive/MyDrive/KLTN_38/KLTN-2026-testingNtraining",
    "/content/drive/MyDrive/KLTN_38 - Thầy Bảo/KLTN-2026-testingNtraining",
    "/content/drive/MyDrive/KLTN_38"
]

SHARED_DRIVE = None
for p in CANDIDATES:
    if os.path.exists(p):
        SHARED_DRIVE = p
        break

if SHARED_DRIVE is None:
    # Tìm nhanh trong Drive nếu tên lối tắt khác
    found = list(Path("/content/drive/MyDrive").rglob("dataset-v2"))
    if found:
        SHARED_DRIVE = str(found[0].parent.parent)

if SHARED_DRIVE is None:
    raise FileNotFoundError(
        "❌ CHƯA TÌM THẤY THƯ MỤC CHUNG!\n"
        "👉 Hãy vào Google Drive web -> Chuột phải vào thư mục chung -> 'Thêm lối tắt vào Drive' -> chọn 'My Drive' rồi chạy lại cell này."
    )

print(f"✅ Đã kết nối thành công với thư mục chung: {SHARED_DRIVE}")

# 3. Clone repo mã nguồn
!rm -rf /content/klcn2026
!git clone https://github.com/mizzhau/yolo11n-cbam-mvtec-defect-detection.git /content/klcn2026
%cd /content/klcn2026
!git checkout main
!git pull origin main

# 4. Cài đặt các thư viện cần thiết
!pip install -q ultralytics albumentations tabulate pyyaml
print("\n🚀 Môi trường huấn luyện đã sẵn sàng!")

## Bước 2️⃣: Giải Nén Dữ Liệu V2 Lên SSD Colab & Cấu Hình Dataset

In [ ]:
import os
import yaml
from pathlib import Path

# ==============================================================================
# 👉 CHỌN TỈ LỆ PHÂN CHIA HUẤN LUYỆN: '70_15_15' | '80_10_10' | '60_20_20'
SELECTED_RATIO = "70_15_15"
# ==============================================================================

# 1. Khởi tạo thư mục lưu kết quả thực nghiệm trên Google Drive (Lưu phẳng trực tiếp)
RUNS_DIR = f"{SHARED_DRIVE}/runs_exp2_{SELECTED_RATIO}"
os.makedirs(f"{RUNS_DIR}/baseline", exist_ok=True)
os.makedirs(f"{RUNS_DIR}/backbone", exist_ok=True)
os.makedirs(f"{RUNS_DIR}/neck", exist_ok=True)
os.makedirs(f"{RUNS_DIR}/head", exist_ok=True)
os.makedirs(f"{RUNS_DIR}/backbone_head", exist_ok=True)
print(f"✅ Thư mục lưu trữ kết quả trên Drive: {RUNS_DIR}")

# 2. Tìm file zip tương ứng trên Drive
zip_hits = list(Path(SHARED_DRIVE).rglob(f"KLTN_prep_v2_{SELECTED_RATIO}.zip"))
if not zip_hits:
    raise FileNotFoundError(f"❌ Không tìm thấy file zip: KLTN_prep_v2_{SELECTED_RATIO}.zip trong {SHARED_DRIVE}")
ZIP_SRC = str(zip_hits[0])
EXTRACT_DIR = f"/content/dataset_v2_{SELECTED_RATIO}"

# 3. Giải nén lên SSD Colab
print(f"📦 Đang giải nén {ZIP_SRC} lên SSD Colab...")
!mkdir -p "{EXTRACT_DIR}"
!unzip -q -o "{ZIP_SRC}" -d "{EXTRACT_DIR}"
print(f"✅ Giải nén hoàn tất vào: {EXTRACT_DIR}")

# 4. Tự động sinh file cấu hình data_colab_active.yaml trỏ đúng danh sách split
DATASET_ROOT = Path(EXTRACT_DIR) / "KLTN_prep_v2" if (Path(EXTRACT_DIR) / "KLTN_prep_v2").exists() else Path(EXTRACT_DIR)

REF_YAML = f"/content/klcn2026/configs/data/exp_2/data_{SELECTED_RATIO}.yaml"
with open(REF_YAML, "r", encoding="utf-8") as f:
    ref_data = yaml.safe_load(f)

colab_yaml_content = {
    "path": str(DATASET_ROOT),
    "train": str(DATASET_ROOT / f"train_{SELECTED_RATIO}.txt"),
    "val": str(DATASET_ROOT / f"val_{SELECTED_RATIO}.txt"),
    "test": str(DATASET_ROOT / f"test_{SELECTED_RATIO}.txt"),
    "nc": ref_data.get("nc", 48),
    "names": ref_data.get("names", [])
}

ACTIVE_YAML_PATH = "/content/klcn2026/configs/data/exp_2/data_colab_active.yaml"
os.makedirs(os.path.dirname(ACTIVE_YAML_PATH), exist_ok=True)
with open(ACTIVE_YAML_PATH, "w", encoding="utf-8") as f:
    yaml.dump(colab_yaml_content, f, allow_unicode=True, sort_keys=False)

print(f"✅ Đã tạo cấu hình YOLO tại: {ACTIVE_YAML_PATH}")

## Bước 3️⃣: Thực Nghiệm Huấn Luyện & Đánh Giá So Sánh

### Cell 3A: Huấn Luyện BASELINE (YOLO11n) 🚀

In [ ]:
!python src/training/train_baseline.py \
    --data configs/data/exp_2/data_colab_active.yaml \
    --epochs 100 \
    --batch 16 \
    --imgsz 640 \
    --save_period 10 \
    --project "{RUNS_DIR}" \
    --name baseline

### Cell 3B: Huấn Luyện CBAM BACKBONE & Đánh Giá So Sánh 🌟

In [ ]:
# 1. Huấn luyện mô hình CBAM Backbone
!python src/training/train_cbam.py \
    --model configs/models/yolo11n_cbam_backbone.yaml \
    --data configs/data/exp_2/data_colab_active.yaml \
    --epochs 100 \
    --batch 16 \
    --imgsz 640 \
    --save_period 10 \
    --project "{RUNS_DIR}" \
    --name backbone

# 2. Xuất biểu đồ và bảng số liệu so sánh trực quan
!python src/evaluation/plot_comparison.py \
    --baseline_dir "{RUNS_DIR}/baseline" \
    --cbam_dir "{RUNS_DIR}/backbone" \
    --output_dir "{RUNS_DIR}/backbone/evaluation"

### Cell 3B_crack: Chạy Tiếp Tục CBAM BACKBONE Nếu Bị Crash Giữa Chừng 🔄
> (Nếu bị ngắt phiên Colab: hãy chạy lại Bước 1 & Bước 2 trước, sau đó chạy thẳng cell này, không chạy lại Cell 3B).

In [ ]:
from src.models.cbam import register_cbam_to_ultralytics
register_cbam_to_ultralytics()
from ultralytics import YOLO

CHECKPOINT = f"{RUNS_DIR}/backbone/weights/last.pt"
model = YOLO(CHECKPOINT)
model.train(resume=True)

### Cell 3C: Huấn Luyện CBAM NECK & Đánh Giá So Sánh 🌟

In [ ]:
# 1. Huấn luyện mô hình CBAM Neck
!python src/training/train_cbam.py \
    --model configs/models/yolo11n_cbam_neck.yaml \
    --data configs/data/exp_2/data_colab_active.yaml \
    --epochs 100 \
    --batch 16 \
    --imgsz 640 \
    --save_period 10 \
    --project "{RUNS_DIR}" \
    --name neck

# 2. Xuất biểu đồ và bảng số liệu so sánh trực quan
!python src/evaluation/plot_comparison.py \
    --baseline_dir "{RUNS_DIR}/baseline" \
    --cbam_dir "{RUNS_DIR}/neck" \
    --output_dir "{RUNS_DIR}/neck/evaluation"

### Cell 3C_crack: Chạy Tiếp Tục CBAM NECK Nếu Bị Crash Giữa Chừng 🔄

In [ ]:
from src.models.cbam import register_cbam_to_ultralytics
register_cbam_to_ultralytics()
from ultralytics import YOLO

CHECKPOINT = f"{RUNS_DIR}/neck/weights/last.pt"
model = YOLO(CHECKPOINT)
model.train(resume=True)

### Cell 3D: Huấn Luyện CBAM HEAD & Đánh Giá So Sánh 🌟

In [ ]:
# 1. Huấn luyện mô hình CBAM Head
!python src/training/train_cbam.py \
    --model configs/models/yolo11n_cbam_head.yaml \
    --data configs/data/exp_2/data_colab_active.yaml \
    --epochs 100 \
    --batch 16 \
    --imgsz 640 \
    --save_period 10 \
    --project "{RUNS_DIR}" \
    --name head

# 2. Xuất biểu đồ và bảng số liệu so sánh trực quan
!python src/evaluation/plot_comparison.py \
    --baseline_dir "{RUNS_DIR}/baseline" \
    --cbam_dir "{RUNS_DIR}/head" \
    --output_dir "{RUNS_DIR}/head/evaluation"

### Cell 3D_crack: Chạy Tiếp Tục CBAM HEAD Nếu Bị Crash Giữa Chừng 🔄

In [ ]:
from src.models.cbam import register_cbam_to_ultralytics
register_cbam_to_ultralytics()
from ultralytics import YOLO

CHECKPOINT = f"{RUNS_DIR}/head/weights/last.pt"
model = YOLO(CHECKPOINT)
model.train(resume=True)

### Cell 3E: Huấn Luyện CBAM BACKBONE + HEAD & Đánh Giá So Sánh 🌟

In [ ]:
# 1. Huấn luyện mô hình CBAM Backbone + Head
!python src/training/train_cbam.py \
    --model configs/models/yolo11n_cbam_backbone_head.yaml \
    --data configs/data/exp_2/data_colab_active.yaml \
    --epochs 100 \
    --batch 16 \
    --imgsz 640 \
    --save_period 10 \
    --project "{RUNS_DIR}" \
    --name backbone_head

# 2. Xuất biểu đồ và bảng số liệu so sánh trực quan
!python src/evaluation/plot_comparison.py \
    --baseline_dir "{RUNS_DIR}/baseline" \
    --cbam_dir "{RUNS_DIR}/backbone_head" \
    --output_dir "{RUNS_DIR}/backbone_head/evaluation"

### Cell 3E_crack: Chạy Tiếp Tục CBAM BACKBONE + HEAD Nếu Bị Crash Giữa Chừng 🔄

In [ ]:
from src.models.cbam import register_cbam_to_ultralytics
register_cbam_to_ultralytics()
from ultralytics import YOLO

CHECKPOINT = f"{RUNS_DIR}/backbone_head/weights/last.pt"
model = YOLO(CHECKPOINT)
model.train(resume=True)